In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.sparse as sp

import anndata as ad
import scanpy as sc
import squidpy as sq

In [ ]:
# ============================ USER PARAMETERS ==============================
# input / output paths
RNA_FILE   = './input/RNA_processed_2nd_brain.h5ad'    # preprocessed brain-2 RNA matrix (from Step_9_1)
PROT_FILE  = './input/brain2_nuclear_int_18prot.h5ad'  # brain-2 nuclear-protein matrix
OUTPUT_DIR = 'output'                                  # folder for saved figures

# 15 nuclear-protein marks measured per cell
MARKS = ['CBP', 'CDK9', 'DAPI', 'H2K119u1', 'H3K27ac', 'H3K4me1', 'H3K4me3',
         'H4K16ac', 'H4K8ac', 'HP1a', 'MECP2', 'Matrin3', 'PolII S5p',
         'SF3a66', 'mH2A1']

# RNA Leiden clustering
RNA_N_NEIGHBORS = 5
RNA_MIN_DIST    = 0.1
RNA_RESOLUTION  = 1.3

# protein subclustering
TARGET_2_CLUSTERS = ['12']                 # RNA leiden cluster(s) to subcluster on protein data
TARGET_2          = '_'.join(TARGET_2_CLUSTERS)   # label used in figure titles / filenames
PROT_N_NEIGHBORS  = 5
PROT_MIN_DIST     = 0.1
PROT_RESOLUTION   = 0.14

# spatial-map rotation (degrees) so the brain-2 section sits upright
ROT2 = -137
# ===========================================================================

In [ ]:
# ---- helper functions -----------------------------------------------------
def protein_cell_ids(path):
    """Cell IDs (obs_names) present in a protein h5ad."""
    N = ad.read_h5ad(path)
    return set(N.obs_names.astype(str))

# cache protein reads so re-running the pooling cells is instant (each file read once)
_PROT_CACHE = {}
def load_prot(path):
    if path not in _PROT_CACHE:
        A = ad.read_h5ad(path)
        A.obs_names = A.obs_names.astype(str)
        _PROT_CACHE[path] = A[:, MARKS].copy()   # keep MARKS (+ their 'counts' layer) only
    return _PROT_CACHE[path]

def rot2(pts):
    """Rotate spatial coords by ROT2 degrees about the brain-2 centroid."""
    t = np.deg2rad(ROT2)
    R = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    center = np.asarray(adata_2nd.obsm['spatial']).mean(0)   # evaluated at call time
    return (np.asarray(pts, float) - center) @ R.T + center

In [ ]:
# preprocessed brain-2 RNA matrix (spatial coords, min_counts filter, normalize + log1p)
adata_2nd = ad.read_h5ad(RNA_FILE)

## Keep only RNA cells that have nuclear-protein data

The RNA matrix covers more cells than were measured for nuclear protein intensity. Subset the
brain-2 RNA matrix to the cells present in its protein file (matched on cell ID / `obs_names`),
so downstream analysis only uses cells with paired RNA + protein.
Protein file: `brain2_nuclear_int_18prot.h5ad`.

In [ ]:
# subset the RNA matrix to cells that also have nuclear-protein data
prot_ids_2nd = protein_cell_ids(PROT_FILE)
adata_2nd.obs_names = adata_2nd.obs_names.astype(str)
n2_before = adata_2nd.n_obs
adata_2nd = adata_2nd[adata_2nd.obs_names.isin(prot_ids_2nd)].copy()
print(f'2nd brain: {n2_before} -> {adata_2nd.n_obs} RNA cells with protein data')

In [ ]:
adata_2nd

In [ ]:
# optional: save the preprocessed, protein-matched brain-2 RNA matrix
# adata_2nd.write('./input/RNA_processed_2nd_brain3.h5ad')

## RNA Leiden clustering

In [ ]:
print("PCA")
sc.tl.pca(adata_2nd, svd_solver="arpack")

In [ ]:
print("neighbors")
sc.pp.neighbors(adata_2nd, n_neighbors=RNA_N_NEIGHBORS, random_state=42)

In [ ]:
print("UMAP")
sc.tl.umap(adata_2nd, min_dist=RNA_MIN_DIST)

In [ ]:
print("Leiden")
sc.tl.leiden(adata_2nd, resolution=RNA_RESOLUTION, random_state=42)

In [ ]:
import matplotlib.patheffects as PathEffects

sc.set_figure_params(figsize=(8, 8))

# Plot UMAP with on-data labels
sc.pl.umap(adata_2nd, color="leiden", legend_loc="on data", size=4, show=False)

ax = plt.gca()

ax.set_frame_on(False)   # removes the box
ax.set_xticks([])        # removes x ticks
ax.set_yticks([])        # removes y ticks
ax.set_xlabel("")        # removes x-axis title
ax.set_ylabel("")        # removes y-axis title
ax.set_title("")

for txt in ax.texts:
    txt.set_fontsize(20)
    txt.set_fontname("Arial")
    txt.set_path_effects([
        PathEffects.withStroke(linewidth=1, foreground='white')  # white outline
    ])

for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig("RNA_umap_Brain_2.pdf", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
sq.pl.spatial_scatter(adata_2nd, shape=None, color="leiden", size=6, library_id="one")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

clusters = list(adata_2nd.obs["leiden"].cat.categories)  # cluster IDs from brain-2 clustering
n_clusters = len(clusters)

ncols = 4
nrows = (n_clusters + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows))
axes = axes.flatten()

for i, cluster in enumerate(clusters):
    ax = axes[i]
    sq.pl.spatial_scatter(
        adata_2nd,
        color="leiden",
        groups=[cluster],
        library_id="one",
        size=2,
        shape=None,
        ax=ax,
        title=f"Cluster {cluster}"
    )

# Hide any extra axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Brain 2 — subclustering using protein data

In [ ]:
# brain-2 cells from the target RNA cluster(s) that also have nuclear-protein data
a6_2 = adata_2nd[adata_2nd.obs['leiden'].isin(TARGET_2_CLUSTERS)].copy()
a6_2.obs_names = a6_2.obs_names.astype(str)
Np2 = load_prot(PROT_FILE)                          # cached read (fast on reruns)
ids2 = a6_2.obs_names.intersection(Np2.obs_names)   # vectorized ID match (no Python loop)

# protein AnnData for those cells (keeps the raw 'counts' layer) + RNA spatial coords
adata_prot_sub2 = Np2[ids2].copy()
adata_prot_sub2.obsm['xy'] = pd.DataFrame(a6_2.obsm['spatial'], index=a6_2.obs_names,
                                          columns=['x', 'y']).loc[adata_prot_sub2.obs_names].values.astype(float)
print(f'brain 2: clusters {TARGET_2_CLUSTERS} = {a6_2.n_obs} RNA cells, {adata_prot_sub2.n_obs} with protein')

In [ ]:
# export: rotated spatial map of the RNA cluster(s) used for subclustering (brain 2)
bg=rot2(adata_2nd.obsm['spatial'])
lab=adata_2nd.obs['leiden'].astype(str).values
cats=list(adata_2nd.obs['leiden'].cat.categories); cols=adata_2nd.uns.get('leiden_colors')
def _clcol(cl,k): return cols[cats.index(cl)] if (cols is not None and cl in cats) else cm.tab10(k%10)

fig,ax=plt.subplots(figsize=(9,9))
ax.scatter(bg[:,0],bg[:,1],s=1,c='0.9',linewidths=0,rasterized=True)
for k,cl in enumerate(TARGET_2_CLUSTERS):
    m=lab==cl
    ax.scatter(bg[m,0],bg[m,1],s=6,color=_clcol(cl,k),linewidths=0,rasterized=True,label=f'RNA cluster {cl} (n={int(m.sum())})')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f'brain 2: RNA clusters {TARGET_2_CLUSTERS} used for subclustering')
ax.legend(markerscale=3, frameon=False, loc='upper left', bbox_to_anchor=(1.01,1))
plt.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(f'{OUTPUT_DIR}/brain2_leiden{TARGET_2}_RNAclusters_spatial.pdf', bbox_inches='tight', dpi=150)
fig.savefig(f'{OUTPUT_DIR}/brain2_leiden{TARGET_2}_RNAclusters_spatial.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# top 4 marker genes of the chosen RNA cluster(s) for subclustering (brain 2)
sc.tl.rank_genes_groups(adata_2nd, groupby='leiden', groups=list(TARGET_2_CLUSTERS),
                        reference='rest', method='wilcoxon')
for cl in TARGET_2_CLUSTERS:
    top4 = [g.strip("'\"") for g in sc.get.rank_genes_groups_df(adata_2nd, group=cl)['names'].head(4)]
    print(f'brain 2 - RNA cluster {cl}: {top4}')

In [ ]:
adata_prot_sub2.uns.pop("log1p", None)
adata_prot_sub2.X = adata_prot_sub2.layers["counts"].copy()
sc.pp.normalize_total(adata_prot_sub2, inplace=True)
sc.pp.log1p(adata_prot_sub2)
adata_prot_sub2.layers["log1p"] = adata_prot_sub2.X.copy()
sc.pp.scale(adata_prot_sub2)
sc.pp.neighbors(adata_prot_sub2, n_neighbors=PROT_N_NEIGHBORS, random_state=42, use_rep="X")
sc.tl.umap(adata_prot_sub2, min_dist=PROT_MIN_DIST)
sc.tl.leiden(adata_prot_sub2, resolution=PROT_RESOLUTION, random_state=42, key_added="protein_subcluster")

# derive plotting variables from the same object so sub2 / X2 / XY2 always align
sub2  = adata_prot_sub2.obs["protein_subcluster"].astype(int).values
_Xl2  = adata_prot_sub2.layers["log1p"]
X2    = _Xl2.toarray() if sp.issparse(_Xl2) else np.asarray(_Xl2)
XY2   = adata_prot_sub2.obsm["xy"]
umap2 = adata_prot_sub2.obsm["X_umap"]
NS2   = len(np.unique(sub2))
print(f"\nprotein sub-states: {NS2}")

In [ ]:
fig,ax=plt.subplots(figsize=(8.5,7.5))
o=np.random.RandomState(0).permutation(len(sub2))
for cc in range(NS2):
    s=sub2[o]==cc
    ax.scatter(umap2[o][s,0],umap2[o][s,1],s=8,color=cm.tab20(cc%20),linewidths=0,rasterized=True,label=f'sub {cc}')
ax.set_title(f'brain 2 — RNA cluster {TARGET_2} -> {NS2} protein sub-states')
ax.legend(markerscale=2,ncol=2,fontsize=8,frameon=False)
ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
plt.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(f'{OUTPUT_DIR}/brain2_leiden{TARGET_2}_subcluster_umap.pdf', bbox_inches='tight'); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(9,9))
bg=rot2(adata_2nd.obsm['spatial']); XY2r=rot2(XY2)
ax.scatter(bg[:,0],bg[:,1],s=1,c='0.9',linewidths=0,rasterized=True)
for cc in range(NS2):
    s=sub2==cc
    ax.scatter(XY2r[s,0],XY2r[s,1],s=9,color=cm.tab20(cc%20),linewidths=0,rasterized=True,label=f'sub {cc}')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f'brain 2: RNA cluster {TARGET_2} -> {NS2} protein sub-states (n={len(sub2)})')
ax.legend(markerscale=3, frameon=False, ncol=2, bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
# fig.savefig(f'{OUTPUT_DIR}/brain2_leiden{TARGET_2}_subclusters_spatial.pdf', bbox_inches='tight', dpi=150)
# fig.savefig(f'{OUTPUT_DIR}/brain2_leiden{TARGET_2}_subclusters_spatial.png', bbox_inches='tight', dpi=150); plt.show()

In [ ]:
# display each sub-state (brain 2) in its own subplot
ncols=3; nrows=(NS2+ncols-1)//ncols
fig,axes=plt.subplots(nrows,ncols,figsize=(4.5*ncols,4.5*nrows))
axes=np.array(axes).reshape(-1)
bg=rot2(adata_2nd.obsm['spatial']); XY2r=rot2(XY2)
for cc in range(NS2):
    ax=axes[cc]
    ax.scatter(bg[:,0],bg[:,1],s=1,c='0.92',linewidths=0,rasterized=True)
    m=sub2==cc
    ax.scatter(XY2r[m,0],XY2r[m,1],s=6,color=cm.tab20(cc%20),linewidths=0,rasterized=True)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    ax.set_title(f'sub {cc} (n={int(m.sum())})',fontsize=10)
for j in range(NS2,len(axes)): axes[j].set_visible(False)
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/brain2_leiden{TARGET_2}_subclusters_separate.pdf', bbox_inches='tight', dpi=150)
# fig.savefig(f'{OUTPUT_DIR}/brain2_leiden{TARGET_2}_subclusters_separate.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
sizes2=pd.Series(sub2).value_counts().sort_index()
prof2=pd.DataFrame(X2, columns=MARKS); prof2['s']=sub2; prof2=prof2.groupby('s')[MARKS].mean()
fig,(a1,a2)=plt.subplots(1,2,figsize=(17,0.45*NS2+2),gridspec_kw={'width_ratios':[1,1.6]})
a1.barh([f'sub {i}' for i in sizes2.index], sizes2.values, color=[cm.tab20(i%20) for i in sizes2.index])
a1.set_xlabel('number of cells'); a1.set_title('sub-state sizes (brain 2)'); a1.invert_yaxis()
im=a2.imshow(((prof2-prof2.mean())/(prof2.std()+1e-9)).values, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
a2.set_xticks(range(len(MARKS))); a2.set_xticklabels(MARKS, rotation=45, ha='right')
a2.set_yticks(range(NS2)); a2.set_yticklabels([f'sub {i}' for i in range(NS2)])
a2.set_title('sub-state mark profiles (z)'); fig.colorbar(im, ax=a2)
plt.tight_layout(); fig.savefig(f'{OUTPUT_DIR}/brain2_leiden{TARGET_2}_substate_profiles.pdf', bbox_inches='tight'); plt.show()